<a href="https://colab.research.google.com/github/Foxokiso/hermes-agent/blob/main/GAY621_TEXT_RENDER_A100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# gay621 text → image (A100)

You paste a description (or Hermes POSTs it). **This runtime's GPU** renders it. Not the local GPU. Not FAL.

| | |
|---|---|
| GPU | Runtime → **A100** + High-RAM |
| Weights | `MyDrive/models/gay621FurryMaleFocus_gay621XLV10.safetensors` (6938040736 bytes) on **this Colab Google account** |
| Out | `/content/outputs` and, if Drive attached, `MyDrive/AI-outputs/text_render/` |
| Hermes | After Run all, paste the `HERMES TUNNEL URL` in chat |

## Drive (the thing that broke last time)

`drive.mount()` only attaches the Google already in the **top-right avatar**. A second account in the OAuth popup → `credential propagation was unsuccessful`. Hotmail Colab cannot see gmail Drive and the reverse.

**Reliable attach:** skip the mount cell. Left sidebar folder icon → **Mount Drive**. Then Run all (mount cell will see it and skip the popup).

Checkpoint must already live on **this** account. Do not re-upload 6.46 GB. If this account does not have the file, copy it once in Drive (between accounts), then Mount Drive.

## Run

1. Avatar = the account that owns the safetensor
2. Connect A100 + High-RAM
3. Sidebar → Mount Drive (preferred) **or** run the Drive cell
4. Runtime → Run all
5. Copy `HERMES TUNNEL URL` to Hermes — then send the text lock in chat
6. Or paste the lock into the last cell and run that cell only


In [ ]:
# 0) GPU gate — A100. Do not continue on CPU.
import torch, sys
print("python", sys.version.split()[0])
assert torch.cuda.is_available(), "No CUDA. Runtime → Change runtime type → A100 GPU."
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / (1024**3)
name = torch.cuda.get_device_name(0)
print(f"GPU: {name}  VRAM: {vram:.1f} GB")
if vram < 15:
    raise SystemExit(
        "Need a real GPU. This box is %.1fGB (%s). Runtime → A100." % (vram, name)
    )
if vram < 24:
    print("WARN: under 24GB. SDXL will run; A100 is the intended runtime.")
else:
    print("A100-class OK.")


In [ ]:
# 1) Deps. No ngrok.
!pip install -q diffusers transformers accelerate safetensors pillow flask
print("deps ok")


In [ ]:
# 2) Drive — same account as the Colab avatar. Sidebar Mount Drive is the reliable path.
import os
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")


def drive_live():
    try:
        return DRIVE.is_dir() and any(DRIVE.iterdir())
    except OSError:
        return False


if drive_live():
    print("Drive already attached (sidebar or prior mount):", DRIVE)
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("MOUNT FAILED:", type(e).__name__, str(e)[:400])
        print()
        print("This is not a model bug. Colab only mounts the Google in the top-right avatar.")
        print("Do not pick a second account in the popup.")
        print("Fix: folder icon (left) → Mount Drive, then re-run THIS cell.")
        print("The 6.46GB file must already be on THIS account at MyDrive/models/")
        raise SystemExit("Drive not attached.")

for p in (DRIVE / "models", DRIVE / "AI-outputs" / "text_render"):
    p.mkdir(parents=True, exist_ok=True)
    print(p, "ok")
print("drive_live", drive_live())


In [ ]:
# 3) Pin gay621. Size-check. Do not glob. Do not download a substitute.
from pathlib import Path

NEED = 6938040736
CKPT = "gay621FurryMaleFocus_gay621XLV10.safetensors"
SRC = Path("/content/drive/MyDrive/models") / CKPT
if not SRC.is_file():
    raise FileNotFoundError(
        "Missing %s on THIS Colab account. Copy the file here once. Do not download a substitute."
        % SRC
    )
src_sz = SRC.stat().st_size
print("Drive ckpt bytes:", src_sz)
if src_sz != NEED:
    print("WARN: size is not the known 6938040736. Loading anyway if it looks like a real file.")
    if src_sz < 1_000_000_000:
        raise SystemExit("File is too small to be gay621. Stop.")
DRIVE_MODEL_PATH = str(SRC)
print("DRIVE_MODEL_PATH", DRIVE_MODEL_PATH)


In [ ]:
# 4) Load pipeline on THIS GPU
import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

print("Loading gay621 SDXL from Drive onto", torch.cuda.get_device_name(0), "...")
pipe = StableDiffusionXLPipeline.from_single_file(
    DRIVE_MODEL_PATH,
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.to("cuda")
try:
    pipe.enable_xformers_memory_efficient_attention()
    print("xformers on")
except Exception as e:
    print("xformers skipped:", type(e).__name__)
print("pipeline ready")


In [ ]:
# 5) Flask API in a BACKGROUND thread. Do not block — the tunnel cell must run after this.
import io, os, uuid, time, threading, base64
from pathlib import Path
from datetime import datetime, timezone
from flask import Flask, request, jsonify, send_file

OUTPUT_DIR = Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUT = Path("/content/drive/MyDrive/AI-outputs/text_render")
try:
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
except Exception:
    DRIVE_OUT = None

GEN_LOCK = threading.Lock()
STATS = {"total": 0, "busy": False}

DEFAULT_NEG = (
    "low quality, blurry, deformed, bad anatomy, extra limbs, extra penises, "
    "watermark, text, censored, female, child, cub, underage, "
    "webbed hands, webbed feet, cartoon, cel shading, flat color, thick outlines, "
    "chibi, sticker"
)


def _run_one(prompt, negative, width, height, steps, cfg, seed):
    if not prompt or not str(prompt).strip():
        raise ValueError("empty prompt")
    gen = torch.Generator("cuda").manual_seed(int(seed)) if int(seed) >= 0 else None
    with torch.inference_mode():
        img = pipe(
            prompt=str(prompt),
            negative_prompt=str(negative or DEFAULT_NEG),
            width=int(width),
            height=int(height),
            num_inference_steps=int(steps),
            guidance_scale=float(cfg),
            generator=gen,
        ).images[0]
    name = "gen_%s_s%s.png" % (uuid.uuid4().hex[:8], seed)
    dest = OUTPUT_DIR / name
    img.save(dest)
    if DRIVE_OUT is not None:
        try:
            img.save(DRIVE_OUT / name)
        except Exception as e:
            print("Drive save skipped:", e)
    STATS["total"] += 1
    return dest, img


app = Flask(__name__)


@app.after_request
def _cors(resp):
    resp.headers["Access-Control-Allow-Origin"] = "*"
    resp.headers["Access-Control-Allow-Headers"] = "Content-Type"
    resp.headers["Access-Control-Allow-Methods"] = "GET,POST,OPTIONS"
    return resp


@app.route("/health")
def health():
    props = torch.cuda.get_device_properties(0)
    return jsonify(
        {
            "status": "ok",
            "model": "gay621-sdxl",
            "gpu": torch.cuda.get_device_name(0),
            "vram_gb": round(props.total_memory / (1024**3), 1),
            "queue_depth": 1 if STATS["busy"] else 0,
            "processing": STATS["busy"],
            "total_generations": STATS["total"],
            "drive_out": str(DRIVE_OUT) if DRIVE_OUT else None,
        }
    )


@app.route("/generate", methods=["POST", "OPTIONS"])
def generate_one():
    if request.method == "OPTIONS":
        return ("", 204)
    d = request.json or {}
    with GEN_LOCK:
        STATS["busy"] = True
        try:
            path, img = _run_one(
                d.get("prompt", ""),
                d.get("negative_prompt") or DEFAULT_NEG,
                d.get("width", 832),
                d.get("height", 1216),
                d.get("steps", 30),
                d.get("cfg_scale", 7.0),
                d.get("seed", -1),
            )
        finally:
            STATS["busy"] = False
    return send_file(path, mimetype="image/png")


@app.route("/generate/batch", methods=["POST", "OPTIONS"])
def generate_batch():
    if request.method == "OPTIONS":
        return ("", 204)
    d = request.json or {}
    count = max(1, min(int(d.get("count", 1)), 50))
    seed0 = int(d.get("seed", -1))
    results = []
    with GEN_LOCK:
        STATS["busy"] = True
        try:
            for i in range(count):
                seed = (seed0 + i) if seed0 >= 0 else -1
                path, img = _run_one(
                    d.get("prompt", ""),
                    d.get("negative_prompt") or DEFAULT_NEG,
                    d.get("width", 832),
                    d.get("height", 1216),
                    d.get("steps", 30),
                    d.get("cfg_scale", 7.0),
                    seed,
                )
                buf = io.BytesIO()
                img.save(buf, format="PNG")
                results.append(
                    {
                        "index": i,
                        "seed": seed,
                        "filename": path.name,
                        "base64": base64.b64encode(buf.getvalue()).decode("ascii"),
                    }
                )
        finally:
            STATS["busy"] = False
    return jsonify({"results": results, "count": len(results)})


def _serve():
    app.run(host="0.0.0.0", port=5000, threaded=True, use_reloader=False)


threading.Thread(target=_serve, daemon=True).start()
time.sleep(2)
print("Flask on :5000 (background). Next cell opens the tunnel.")


In [ ]:
# 6) Cloudflare quick tunnel. Paste HERMES TUNNEL URL in chat. No ngrok.
import os, re, time, subprocess, threading
from datetime import datetime, timezone
from pathlib import Path

bin_path = "/content/cloudflared"
if not os.path.exists(bin_path):
    subprocess.run(
        [
            "wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", bin_path,
        ],
        check=True,
    )
    os.chmod(bin_path, 0o755)

proc = subprocess.Popen(
    [bin_path, "tunnel", "--url", "http://127.0.0.1:5000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

TUNNEL_URL = None


def read_tunnel():
    global TUNNEL_URL
    for line in proc.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m and TUNNEL_URL is None:
            TUNNEL_URL = m.group(0)
            print("=" * 60)
            print("HERMES TUNNEL URL:", TUNNEL_URL)
            print("Paste that URL in chat. Then send the text lock.")
            print("=" * 60)
        print(line, end="")


threading.Thread(target=read_tunnel, daemon=True).start()
for _ in range(45):
    if TUNNEL_URL:
        break
    time.sleep(1)

print("TUNNEL_URL:", TUNNEL_URL)
if TUNNEL_URL:
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    try:
        Path("/content/drive/MyDrive/tunnel_url.txt").write_text(
            stamp + "\n" + TUNNEL_URL + "\n", encoding="utf-8"
        )
        print("Wrote MyDrive/tunnel_url.txt")
    except Exception as e:
        print("Drive tunnel write skipped:", e, "— paste the URL, do not wait on Drive.")
else:
    print("No tunnel yet. Flask is still on 127.0.0.1:5000 inside Colab. Re-run THIS cell only.")


In [ ]:
# 7) OPTIONAL — paste a lock here and run this cell. Or skip and let Hermes POST.
# Re-run this cell only for another batch. Skip-if-exists by seed.

COUNT = 4
SEED0 = 270913
WIDTH, HEIGHT = 832, 1216
STEPS = 30
CFG = 7.0
PREFIX = "text_render"

# Paste the scene/character here. Empty = do nothing (tunnel path).
DEFAULT_PROMPT = ""

NEG = DEFAULT_NEG  # from Flask cell

from pathlib import Path
import torch, time, json
from datetime import datetime, timezone

OUT = Path("/content/outputs")
OUT.mkdir(parents=True, exist_ok=True)
DRIVE_OUT = Path("/content/drive/MyDrive/AI-outputs/text_render")
try:
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
except Exception:
    DRIVE_OUT = None

prompt = DEFAULT_PROMPT.strip()
if not prompt:
    print("No DEFAULT_PROMPT. API is live — paste HERMES TUNNEL URL in chat and send the text there.")
    print("Or paste a lock into DEFAULT_PROMPT and re-run this cell.")
else:
    meta = {
        "prompt": prompt,
        "count": COUNT,
        "seed0": SEED0,
        "utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }
    (OUT / "prompt.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    saved = skipped = 0
    for i in range(COUNT):
        seed = SEED0 + i
        name = "%s_%02d_s%s.png" % (PREFIX, i, seed)
        dest = OUT / name
        if dest.is_file() and dest.stat().st_size > 200000:
            print("skip", name)
            skipped += 1
            continue
        t0 = time.time()
        gen = torch.Generator("cuda").manual_seed(seed)
        with torch.inference_mode():
            img = pipe(
                prompt=prompt,
                negative_prompt=NEG,
                width=WIDTH,
                height=HEIGHT,
                num_inference_steps=STEPS,
                guidance_scale=CFG,
                generator=gen,
            ).images[0]
        img.save(dest)
        if DRIVE_OUT is not None:
            img.save(DRIVE_OUT / name)
        print("saved", name, dest.stat().st_size, "%.1fs" % (time.time() - t0))
        saved += 1
        torch.cuda.empty_cache()
    print("done saved=%s skipped=%s out=%s" % (saved, skipped, OUT))
